# AutoDDG baseline on PANGAEA (SURF/Willma)

Run the AutoDDG description pipeline on every CSV under `data/experiment/pangea` (fallback: `pangaea_datasets`) using the SURF LLM, not OpenAI.

This produces free-text AutoDDG descriptions (topic + profile + semantic analysis + description), not Croissant metadata. Outputs go to `outputs/autoddg_pangaea_<model_slug>/`.

Uses SURF/Willma (`LLM_PROVIDER=surf`) with `Qwen/Qwen3.6-35B-A3B-FP8`, matching the M³ PANGAEA run in `outputs/pangaea_Qwen_3.6_35B/`. Change `MODEL_OVERRIDE` if you want another sequence.

Each CSV has a 10-minute wall-clock limit. Timeouts are written to `{id}_autoddg.skipped.json` and `run_log.jsonl` in the output folder, and are skipped on resume along with successful files.

Run the setup cell, then the smoke test, then the batch loop.

In [ ]:
import json
import logging
import os
import re
import sys
import time
import traceback
from pathlib import Path

from dotenv import load_dotenv

BASE = Path('../..').resolve()
sys.path.insert(0, str(BASE))

load_dotenv(BASE / '.env')
os.environ['LLM_PROVIDER'] = 'surf'
# Optional override; leave None to use LLM_MODEL from .env
MODEL_OVERRIDE = 'mistralai/Mistral-Small-3.2-24B-Instruct-2506'
if MODEL_OVERRIDE:
    os.environ['LLM_MODEL'] = MODEL_OVERRIDE

from src.baseline.autoddg.csv_runner import describe_csv_with_timeout
from src.config import LLM_PROVIDER, get_model_name

LLM_NAME = get_model_name()
LLM_SLUG = re.sub(r'[^\w.\-]+', '_', LLM_NAME.replace('/', '_'))
EXPERIMENT_RUN = f'autoddg_pangaea_{LLM_SLUG}'

SAMPLE_SIZE = 20
DESCRIPTION_WORDS = 100
RANDOM_STATE = 9
PROCESS_TIMEOUT_SECONDS = 600

PANGEA_DIR_CANDIDATES = [
    BASE / 'data/experiment/pangea',
    BASE / 'data/experiment/pangaea_datasets',
]


def resolve_pangea_dir() -> Path:
    for path in PANGEA_DIR_CANDIDATES:
        if path.is_dir():
            return path
    searched = '\n  '.join(str(p) for p in PANGEA_DIR_CANDIDATES)
    raise FileNotFoundError(f'PANGAEA data directory not found. Tried:\n  {searched}')


def collect_pangea_csv_files(data_dir: Path) -> list[Path]:
    return sorted(data_dir.glob('*.csv'))


def output_path_for(csv_path: Path) -> Path:
    return OUTPUT_DIR / f'{csv_path.stem}_autoddg.json'


def skip_path_for(csv_path: Path) -> Path:
    return OUTPUT_DIR / f'{csv_path.stem}_autoddg.skipped.json'


def is_processed(csv_path: Path) -> bool:
    return output_path_for(csv_path).exists() or skip_path_for(csv_path).exists()


PANGEA_DIR = resolve_pangea_dir()
OUTPUT_DIR = BASE / 'outputs' / EXPERIMENT_RUN
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

RUN_LOG = OUTPUT_DIR / 'run_log.jsonl'

all_csv_files = collect_pangea_csv_files(PANGEA_DIR)
if not all_csv_files:
    raise FileNotFoundError(f'No CSV files found in {PANGEA_DIR}')

csv_files = [p for p in all_csv_files if not is_processed(p)]
skipped_count = len(all_csv_files) - len(csv_files)

print(f'LLM provider: {LLM_PROVIDER}')
print(f'LLM model: {LLM_NAME}')
print(f'Experiment run: {EXPERIMENT_RUN}')
print(f'Per-file timeout: {PROCESS_TIMEOUT_SECONDS}s')
print(f'Data directory: {PANGEA_DIR}')
print(f'Found {len(all_csv_files)} CSV file(s)')
print(f'Skipping {skipped_count} already completed or timed out in {OUTPUT_DIR}')
print(f'Remaining to run: {len(csv_files)}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Run log: {RUN_LOG}')

LLM provider: surf
LLM model: Sehyo/Qwen3.5-122B-A10B-NVFP4
Experiment run: autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4
Per-file timeout: 600s
Data directory: /home/com3dian/Github/metadata_agent/data/experiment/pangaea_datasets
Found 75 CSV file(s)
Skipping 0 already completed or timed out in /home/com3dian/Github/metadata_agent/outputs/autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4
Remaining to run: 75
Output directory: /home/com3dian/Github/metadata_agent/outputs/autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4
Run log: /home/com3dian/Github/metadata_agent/outputs/autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4/run_log.jsonl


In [2]:
def append_run_log(event: dict) -> None:
    payload = {
        'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
        **event,
    }
    with open(RUN_LOG, 'a', encoding='utf-8') as f:
        f.write(json.dumps(payload, ensure_ascii=False) + '\n')


def write_timeout_skip(csv_path: Path, message: str) -> tuple[Path, dict]:
    record = {
        'status': 'skipped',
        'reason': 'timeout',
        'csv': csv_path.name,
        'dataset_id': csv_path.stem,
        'provider': LLM_PROVIDER,
        'model': LLM_NAME,
        'timeout_seconds': PROCESS_TIMEOUT_SECONDS,
        'runtime_seconds': PROCESS_TIMEOUT_SECONDS,
        'message': message,
    }
    skip_file = skip_path_for(csv_path)
    with open(skip_file, 'w', encoding='utf-8') as f:
        json.dump(record, f, indent=2, ensure_ascii=False)
    append_run_log({**record, 'skip_file': str(skip_file)})
    return skip_file, record


def run_autoddg(csv_path: Path) -> dict:
    """Run AutoDDG with a 10-minute process timeout. Writes a skip file on timeout."""
    try:
        return describe_csv_with_timeout(
            csv_path,
            timeout_seconds=PROCESS_TIMEOUT_SECONDS,
            sample_size=SAMPLE_SIZE,
            description_words=DESCRIPTION_WORDS,
            random_state=RANDOM_STATE,
            model_name=LLM_NAME,
        )
    except TimeoutError as exc:
        skip_file, record = write_timeout_skip(csv_path, str(exc))
        print(f'  TIMEOUT after {PROCESS_TIMEOUT_SECONDS}s: {csv_path.name}')
        print(f'  Skip record: {skip_file}')
        return record

## Smoke test

Run AutoDDG on the first remaining CSV (or the first file if all are done). Successful results and timeout skips are both saved in the output folder so the batch cell will not retry them.

In [3]:
smoke_path = csv_files[0] if csv_files else all_csv_files[0]
print(f'Smoke test: {smoke_path.name}')

if is_processed(smoke_path):
    existing = output_path_for(smoke_path)
    if not existing.exists():
        existing = skip_path_for(smoke_path)
    with open(existing, encoding='utf-8') as f:
        smoke_result = json.load(f)
    print(f'Already processed: {existing}')
else:
    smoke_result = run_autoddg(smoke_path)
    if smoke_result.get('status') == 'ok':
        smoke_output = output_path_for(smoke_path)
        with open(smoke_output, 'w', encoding='utf-8') as f:
            json.dump(smoke_result, f, indent=2, ensure_ascii=False)
        print(f'Saved: {smoke_output}')

if smoke_result.get('status') == 'skipped':
    print(f"status=skipped reason={smoke_result.get('reason')}")
    print(smoke_result.get('message'))
else:
    print(
        f"rows={smoke_result.get('n_rows')} cols={smoke_result.get('n_columns')} "
        f"runtime={smoke_result.get('runtime_seconds')}s"
    )
    print(f"topic: {smoke_result.get('topic')}")
    print('\n--- description ---')
    print(smoke_result.get('description'))

Smoke test: 897882.csv


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


Saved: /home/com3dian/Github/metadata_agent/outputs/autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4/897882_autoddg.json
rows=101 cols=20 runtime=46.646s
topic: Oyster Reproductive Traits

--- description ---
This dataset presents detailed reproductive and physiological measurements of the oyster species E. fimbriata collected from three coastal locations in Senegal during 2014. It includes environmental variables such as temperature and salinity alongside biological metrics like oocyte dry mass, volume, and biochemical composition including lipids and proteins. The data also features geochemical ratios of barium, strontium, and zinc relative to calcium, which serve as proxies for environmental conditions. Researchers can utilize this resource to analyze seasonal reproductive cycles, assess the impact of environmental factors on oyster gametogenesis, and study the bioaccumulation of trace elements in marine bivalves across different spawning batches.


## Batch

Loop remaining PANGAEA CSVs. Successful outputs and timeout skip records are both treated as done on the next run. Other failures are retried.

In [4]:
csv_files = [p for p in all_csv_files if not is_processed(p)]
skipped_count = len(all_csv_files) - len(csv_files)
print(f'Remaining to run: {len(csv_files)} (already done or timed out: {skipped_count})')

results_summary = []
success_count = 0
timeout_count = 0

for i, csv_path in enumerate(csv_files, start=1):
    print(f'\n{"=" * 60}')
    print(f'[{i}/{len(csv_files)}] {csv_path.name}')
    entry = {
        'csv': csv_path.name,
        'success': False,
        'output_file': None,
        'error': None,
    }
    try:
        result = run_autoddg(csv_path)
        if result.get('status') == 'skipped':
            timeout_count += 1
            entry['error'] = result.get('message')
            entry['output_file'] = str(skip_path_for(csv_path))
            print(f"  SKIPPED (timeout) -> {skip_path_for(csv_path).name}")
        else:
            output_file = output_path_for(csv_path)
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(result, f, indent=2, ensure_ascii=False)
            entry['success'] = True
            entry['output_file'] = str(output_file)
            success_count += 1
            print(
                f"  topic={result['topic']!r} "
                f"runtime={result['runtime_seconds']}s -> {output_file.name}"
            )
    except Exception as exc:
        entry['error'] = str(exc)
        print(f'  FAILED: {exc}')
        traceback.print_exc()
    results_summary.append(entry)

print(f'\n{"=" * 60}')
print(f'Batch complete: {success_count}/{len(csv_files)} successful this run')
print(f'Timed out this run: {timeout_count}')
print(f'Previously completed or timed out (skipped): {skipped_count}')
print(
    f'Total with outputs or skip records: '
    f'{skipped_count + success_count + timeout_count}/{len(all_csv_files)}'
)
for item in results_summary:
    status = 'OK' if item['success'] else 'FAIL'
    detail = item['output_file'] or item.get('error') or 'unknown error'
    print(f'  [{status}] {item["csv"]} -> {detail}')

Remaining to run: 74 (already done or timed out: 1)

[1/74] 899705.csv


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


  topic='MITAPI Emissions Data' runtime=8.438s -> 899705_autoddg.json

[2/74] 901488.csv


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


  topic='River Delta Survey' runtime=17.975s -> 901488_autoddg.json

[3/74] 901510.csv


...

  topic='Sediment Geochemistry' runtime=99.457s -> 990164_autoddg.json

[73/74] 995133.csv


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


  topic='Baltic Sea Mussels' runtime=48.116s -> 995133_autoddg.json

[74/74] 995327.csv


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


  topic='Sediment Mineralogy' runtime=71.253s -> 995327_autoddg.json

Batch complete: 74/74 successful this run
Timed out this run: 0
Previously completed or timed out (skipped): 1
Total with outputs or skip records: 75/75
  [OK] 899705.csv -> /home/com3dian/Github/metadata_agent/outputs/autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4/899705_autoddg.json
  [OK] 901488.csv -> /home/com3dian/Github/metadata_agent/outputs/autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4/901488_autoddg.json
  [OK] 901510.csv -> /home/com3dian/Github/metadata_agent/outputs/autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4/901510_autoddg.json
  [OK] 902576.csv -> /home/com3dian/Github/metadata_agent/outputs/autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4/902576_autoddg.json
  [OK] 906209.csv -> /home/com3dian/Github/metadata_agent/outputs/autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4/906209_autoddg.json
  [OK] 908459.csv -> /home/com3dian/Github/metadata_agent/outputs/autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4/90845